In [29]:
import pandas as pd
import os
from rapidfuzz.fuzz import ratio
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

base_path = "archive (4)/student_resource/dataset"
train_path = base_path + "/train"

train_s1 = pd.read_csv(train_path + "/train_source1.tsv", sep="\t")
train_s2 = pd.read_csv(train_path + "/train_source2.tsv", sep="\t")
train_s3 = pd.read_csv(train_path + "/train_source3.tsv", sep="\t")
ground_truth = pd.read_csv(train_path + "/train_ground_truth.tsv", sep="\t")

print(train_s1.shape)
print(train_s2.shape)
print(train_s3.shape)
print(ground_truth.shape)

(2206821, 4)
(5034616, 4)
(5285603, 4)
(2206821, 2)


In [5]:
import sys

print(sys.executable)

c:\Users\jayap\OneDrive\Desktop\AIML_HACK\.venv\Scripts\python.exe


In [6]:

!{sys.executable} -m pip install pandas numpy scikit-learn rapidfuzz kagglehub


[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
import pandas as pd
import numpy as np
import os
import kagglehub
import os
from difflib import SequenceMatcher
print("Setup successful!")

Setup successful!


c:\Users\jayap\OneDrive\Desktop\AIML_HACK\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [9]:
print("Current working folder:")
print(os.getcwd())

print("\nFiles/folders here:")
print(os.listdir("."))

Current working folder:
c:\Users\jayap\OneDrive\Desktop\AIML_HACK

Files/folders here:
['.venv', 'amazon_matching.ipynb', 'archive (4)']


In [10]:
print("Current working folder:")
print(os.getcwd())

print("\nFiles/folders here:")
print(os.listdir("."))

Current working folder:
c:\Users\jayap\OneDrive\Desktop\AIML_HACK

Files/folders here:
['.venv', 'amazon_matching.ipynb', 'archive (4)']


In [11]:
print(os.listdir("archive (4)"))

['student_resource']


In [12]:
base_path = "archive (4)/student_resource/dataset"

print("Dataset path:", base_path)
print("Folders:", os.listdir(base_path))

Dataset path: archive (4)/student_resource/dataset
Folders: ['test', 'train']


In [13]:
train_path = base_path + "/train"

print("Train files:")
print(os.listdir(train_path))

Train files:
['train_ground_truth.tsv', 'train_source1.tsv', 'train_source2.tsv', 'train_source3.tsv']


In [14]:
train_s1 = pd.read_csv(
    train_path + "/train_source1.tsv",
    sep="\t"
)

train_s2 = pd.read_csv(
    train_path + "/train_source2.tsv",
    sep="\t"
)

train_s3 = pd.read_csv(
    train_path + "/train_source3.tsv",
    sep="\t"
)

ground_truth = pd.read_csv(
    train_path + "/train_ground_truth.tsv",
    sep="\t"
)

print("Source 1:", train_s1.shape)
print("Source 2:", train_s2.shape)
print("Source 3:", train_s3.shape)
print("Ground Truth:", ground_truth.shape)

Source 1: (2206821, 4)
Source 2: (5034616, 4)
Source 3: (5285603, 4)
Ground Truth: (2206821, 2)


In [15]:
print("Source 1:", train_s1.shape)
print("Source 2:", train_s2.shape)
print("Source 3:", train_s3.shape)
print("Ground Truth:", ground_truth.shape)

print("\nSource 1 columns:")
print(train_s1.columns.tolist())

print("\nGround Truth columns:")
print(ground_truth.columns.tolist())

Source 1: (2206821, 4)
Source 2: (5034616, 4)
Source 3: (5285603, 4)
Ground Truth: (2206821, 2)

Source 1 columns:
['entity_id', 'business_name', 'business_address', 'country']

Ground Truth columns:
['source1_entity_id', 'matched_entity_ids']


In [16]:
print("========== SOURCE 1 ==========")
display(train_s1.head())

print("\n========== SOURCE 2 ==========")
display(train_s2.head())

print("\n========== SOURCE 3 ==========")
display(train_s3.head())

print("\n========== GROUND TRUTH ==========")
display(ground_truth.head())

========== SOURCE 1 ==========


,entity_id,business_name,business_address,country
0,S1-925783039,Orelee's Barbershop,"1795 Westchester Drive, High Point, NC",US
1,S1-773889195,Prime Money,"17560 Ellis Road, Tahlequah, OK",US
2,S1-377745466,B+ Retail Inc,"1712 Montebello Avenue, Phoenix, AZ",US
3,S1-133037285,Christ Chapel,"2100 Cameron Drive, Unit APARTMENT G, Dundalk, MD",US
4,S1-755362802,Prabhav Business Center,"797, Lake Town Block A, Kolkata, Howrah, West ...",India



========== SOURCE 2 ==========


,entity_id,business_name,business_address,country
0,S2-166376419,राम मार्केटिंग प्राइवेट लिमिटेड,"KH NO. -570/13, NEW DELHI, WEST DELHI, Delhi",India
1,S2-764573417,-- Holloway Peak Inc Seafood,"105 ELM ST, MORGANTON, NC",US
2,S2-639257739,आदित्य प्रॉपर्टीज एलएलपी,"G-3/571, GULMOHAR COLONY, BHOPAL, Madhya Pradesh",India
3,S2-163963287,Summit Inc,"GREENSBORO, NC, 19 1/2 STARDUST TRAIL",US
4,S2-49942811,Delta Tetlecommunication Inc,"914 PIERPONT AVE, CLEVELAND, OH",US



========== SOURCE 3 ==========


,entity_id,business_name,business_address,country
0,S3-202863386,wilfordhancock.com,"Mack Rd, Haltom City, Texas",US
1,S3-859268022,International South Consultants Private Ltd,NaN,India
2,S3-22467283,LLC Moncada Léarning Center,"5780 Fawn Ct, Fort Worth, Texas",US
3,S3-671162755,Moyna's Coffee,"1 Ivanhoe Ave, PO Box 6009, Cincinnati, Ohio",US
4,S3-960981775,Pvt. EFS Print Ventures Ltd.,"Door No 183, 41St Cross, 22Nd Main 9Th Block J...",India



========== GROUND TRUTH ==========


,source1_entity_id,matched_entity_ids
0,S1-965667,"S2-681193310,S2-743505751,S3-775321672,S3-1129..."
1,S1-55344266,"S2-249013014,S2-197070651,S3-478195123,S3-3843..."
2,S1-343815751,"S2-790675320,S2-479876582,S3-878454467"
3,S1-656753428,"S2-153058913,S2-24659151,S3-679606215"
4,S1-102811957,"S2-478959098,S2-553508714,S2-625774905,S3-7280..."


In [17]:
# Sample 10,000 Source-1 records

sample_s1 = train_s1.sample(
    n=10000,
    random_state=42
).copy()

print("Sampled S1 records:", len(sample_s1))

# Find corresponding ground-truth rows
sample_ground_truth = ground_truth[
    ground_truth["source1_entity_id"].isin(
        sample_s1["entity_id"]
    )
].copy()

print("Ground-truth rows found:", len(sample_ground_truth))

# Count S1 records with / without matches
match_counts = (
    sample_ground_truth["matched_entity_ids"]
    .fillna("")
    .astype(str)
    .str.strip()
    .ne("")
)

print(
    "S1 records with matches:",
    match_counts.sum()
)

print(
    "S1 records without matches:",
    (~match_counts).sum()
)

Sampled S1 records: 10000
Ground-truth rows found: 10000
S1 records with matches: 9428
S1 records without matches: 572


In [18]:
# Create positive training pairs

positive_pairs = []

# Create quick lookup dictionaries
s1_lookup = train_s1.set_index("entity_id")
s2_lookup = train_s2.set_index("entity_id")
s3_lookup = train_s3.set_index("entity_id")

for _, gt_row in sample_ground_truth.iterrows():

    s1_id = gt_row["source1_entity_id"]

    # Get S1 record
    if s1_id not in s1_lookup.index:
        continue

    s1_row = s1_lookup.loc[s1_id]

    # Get matched entity IDs
    matched_ids = str(gt_row["matched_entity_ids"])

    if matched_ids == "nan" or not matched_ids.strip():
        continue

    matched_ids = [
        x.strip()
        for x in matched_ids.split(",")
        if x.strip()
    ]

    # Create one positive pair for every matched ID
    for candidate_id in matched_ids:

        if candidate_id.startswith("S2-"):
            if candidate_id not in s2_lookup.index:
                continue
            candidate_row = s2_lookup.loc[candidate_id]

        elif candidate_id.startswith("S3-"):
            if candidate_id not in s3_lookup.index:
                continue
            candidate_row = s3_lookup.loc[candidate_id]

        else:
            continue

        positive_pairs.append({
            "s1_id": s1_id,
            "candidate_id": candidate_id,
            "s1_name": s1_row["business_name"],
            "candidate_name": candidate_row["business_name"],
            "s1_address": s1_row["business_address"],
            "candidate_address": candidate_row["business_address"],
            "s1_country": s1_row["country"],
            "candidate_country": candidate_row["country"],
            "label": 1
        })

positive_pairs_df = pd.DataFrame(positive_pairs)

print("Positive pairs:", len(positive_pairs_df))

print("\nFirst 10 positive pairs:")
display(positive_pairs_df.head(10))

Positive pairs: 34768

First 10 positive pairs:


,s1_id,candidate_id,s1_name,candidate_name,s1_address,candidate_address,s1_country,candidate_country,label
0,S1-359440945,S2-683352932,Daon Continental Music LLC,Daon Continental Music LLC,"7667 136, Terrell, TX","7667C 136, TERRELL, TX",US,US,1
1,S1-359440945,S2-940771150,Daon Continental Music LLC,Daon Continental Music,"7667 136, Terrell, TX","7667C 136, TERRELL, TX",US,US,1
2,S1-359440945,S3-75226047,Daon Continental Music LLC,daoncontinentalmusiccom,"7667 136, Terrell, TX","7667 136, Terrell, Texas",US,US,1
3,S1-359440945,S3-137202105,Daon Continental Music LLC,Daon Continental,"7667 136, Terrell, TX","7667 136, Terrell, Texas",US,US,1
4,S1-359440945,S3-20852064,Daon Continental Music LLC,Daon Cóntinental Music Llc,"7667 136, Terrell, TX","136, Terrell, Texas",US,US,1
5,S1-708400226,S2-11447624,Trinity Lutheran Church,Trinity Lutheran Chúrch,"1023 Dakota Street, Spring Valley, IL","SPRING VALLEY, IL, 001023 DAKOTA ST",US,US,1
6,S1-708400226,S2-737021373,Trinity Lutheran Church,TRINITY LUTSHRN CHURCH,"1023 Dakota Street, Spring Valley, IL","DAKOTA ST, SPRING VALLEY, IL",US,US,1
7,S1-708400226,S2-636078522,Trinity Lutheran Church,Trinity Lutheran Church Inc.,"1023 Dakota Street, Spring Valley, IL",NaN,US,US,1
8,S1-708400226,S3-952283138,Trinity Lutheran Church,Trinity Lutheran Church LLC,"1023 Dakota Street, Spring Valley, IL","1023 Dakota Street, Spring Valley, Illinois",US,US,1
9,S1-708400226,S3-654593788,Trinity Lutheran Church,Trinity Lutheran Church,"1023 Dakota Street, Spring Valley, IL","1023 Dakota Street, Spring Valley, Illinois",US,US,1


In [ ]:

# FAST NEGATIVE PAIR GENERATION

import pandas as pd
import numpy as np


# 1. Number of negative pairs we need


positive_count = len(positive_pairs_df)

# Keep approximately 1:1 positive and negative pairs
negative_target = positive_count

print("Positive pairs:", positive_count)
print("Negative pairs required:", negative_target)



# 2. Create a compact candidate pool

# We don't need millions of rows for negative training examples.
# Randomly sample a manageable number from S2 and S3.

SAMPLE_POOL_SIZE = 100000

s2_pool_size = min(SAMPLE_POOL_SIZE, len(train_s2))
s3_pool_size = min(SAMPLE_POOL_SIZE, len(train_s3))

s2_pool = train_s2.sample(
    n=s2_pool_size,
    random_state=42
).copy()

s3_pool = train_s3.sample(
    n=s3_pool_size,
    random_state=42
).copy()

print("\nS2 candidate pool:", len(s2_pool))
print("S3 candidate pool:", len(s3_pool))



# 3. Combine S2 + S3 candidate pool


candidate_pool = pd.concat(
    [
        s2_pool,
        s3_pool
    ],
    ignore_index=True
)

print("Combined candidate pool:", len(candidate_pool))



# 4. Remove IDs that are already positive matches


positive_candidate_ids = set(
    positive_pairs_df["candidate_id"].astype(str)
)

candidate_pool["entity_id"] = (
    candidate_pool["entity_id"].astype(str)
)

negative_candidate_pool = candidate_pool[
    ~candidate_pool["entity_id"].isin(
        positive_candidate_ids
    )
].copy()

print(
    "Available negative candidates:",
    len(negative_candidate_pool)
)


# 5. Randomly select required negative candidates


negative_count = min(
    negative_target,
    len(negative_candidate_pool)
)

negative_candidates = negative_candidate_pool.sample(
    n=negative_count,
    random_state=42
).copy()

print(
    "Selected negative candidates:",
    len(negative_candidates)
)



# 6. Randomly select S1 records


# We create a random S1 pool of the same size.
# This avoids a 10,000-row Python loop.

s1_negative = sample_s1.sample(
    n=negative_count,
    random_state=42,
    replace=True
).reset_index(drop=True)



# 7. Build negative pairs using vectorized operations
negative_candidates = (
    negative_candidates
    .reset_index(drop=True)
)

negative_pairs_df = pd.DataFrame({
    "s1_id": s1_negative["entity_id"].values,

    "candidate_id":
        negative_candidates["entity_id"].values,

    "s1_name":
        s1_negative["business_name"].values,

    "candidate_name":
        negative_candidates["business_name"].values,

    "s1_address":
        s1_negative["business_address"].values,

    "candidate_address":
        negative_candidates["business_address"].values,

    "s1_country":
        s1_negative["country"].values,

    "candidate_country":
        negative_candidates["country"].values,

    "label": 0
})



# 8. Show result


print("\n========================================")
print("NEGATIVE PAIR GENERATION COMPLETE")
print("========================================")

print("Negative pairs:", len(negative_pairs_df))

print("\nFirst 10 negative pairs:")

display(
    negative_pairs_df.head(10)
)

Positive pairs: 34768
Negative pairs required: 34768

S2 candidate pool: 100000
S3 candidate pool: 100000
Combined candidate pool: 200000
Available negative candidates: 199318
Selected negative candidates: 34768

NEGATIVE PAIR GENERATION COMPLETE
Negative pairs: 34768

First 10 negative pairs:


,s1_id,candidate_id,s1_name,candidate_name,s1_address,candidate_address,s1_country,candidate_country,label
0,S1-360994973,S2-48877567,Franks Global Blockchain L.L.C.,Dr Bhudev It Private,"NC, Wilkesboro, 1012 Woodland Boulevard","NO 9TH FLOOR, 2 KAPOORTHALA COMPLEX, ALIGANJ, ...",US,India,0
1,S1-701583396,S2-253499985,Shivam Builders,"NOWAK, BOYNE & LUERA SERIES INC","Flat No. N1, Mathosree Apartments, Plotno.376,...","1475 WILLIAMSON ROAD, TN, MILLERSVILLE",India,US,0
2,S1-530787440,S2-71910369,Bangalore Realty Private Limited,Software Meghana Hóldings,"Survey No. 93/04, Near Matkal Hotel Ksca Stadi...","55, FIRST FLOOR, PATCHAI AMMBN STREET DEVI NAG...",India,India,0
3,S1-261315820,S2-495172279,Crown Web Private Limited,சக்தி இன்ஃபோடெக் பிரைவேட் லிமிடெட்,"Hyderabad, Plot No 22, Telangana, Risala Bazar...","Tamil Nadu, CHENNAI, DOOR NO 144 C2, II FLOOR,...",India,India,0
4,S1-328457260,S3-234942192,Thane Tools Private Limited,Beth Baptist-Church,"Ithink Techno Campus, 7Th Floor, A Wing, Off P...","1016 State Highway 15, Hutchinson, Minnesota",India,US,0
5,S1-387200680,S3-579427453,Guru Consulting Private Limited,The Spitler & Hyer Rail LLC,"Flat No.303, Infinite Heights, Beta Block, Pra...","11 1/2 Colonial Rd, Westport, Connecticut",India,US,0
6,S1-144525073,S3-173254632,Jalandhar Experts Pvt Ltd,Y+ Inc. Service,"Industrial Plot No. 28, Kapurthala Road, Waria...","8352 Fieldpoppy Cir, Sacramento, California",India,US,0
7,S1-352920630,S3-451960065,Health Creative Fund,Royal Group North Center,"New London, NC, 38007 Trinity Circle","Mansfield, Texas, 633 Jamie Ln",US,US,0
8,S1-169200108,S3-65757971,Premier Mobile Inc.,ভারত ইনফ্রাস্ট্রাকচার,"33 Beech Mt Circle, CT, Mansfield","58F, Kali Krishna Tagore Street, 2Nd Floor, Ko...",US,India,0
9,S1-974156317,S3-980661600,Big Pizza,Molina and Berns Partners,"347 89 Street, Brooklyn, NY","5864 26th Avenue, Moline, Illinois",US,US,0


In [21]:
# Combine positive and negative pairs

training_pairs = pd.concat(
    [
        positive_pairs_df,
        negative_pairs_df
    ],
    ignore_index=True
)

# Shuffle the pairs
training_pairs = training_pairs.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

print("Total training pairs:", len(training_pairs))

print("\nLabel distribution:")
print(training_pairs["label"].value_counts())

print("\nPositive pairs:", (training_pairs["label"] == 1).sum())
print("Negative pairs:", (training_pairs["label"] == 0).sum())

print("\nFirst 10 training pairs:")
display(training_pairs.head(10))

Total training pairs: 69536

Label distribution:
label
1    34768
0    34768
Name: count, dtype: int64

Positive pairs: 34768
Negative pairs: 34768

First 10 training pairs:


,s1_id,candidate_id,s1_name,candidate_name,s1_address,candidate_address,s1_country,candidate_country,label
0,S1-429266588,S2-777032501,Surya Marketing Private Limited,सूर्य मार्केटिंग प्राइवेट लिमिटेड,"R.S No 87/2, Ram Nagar Vadgaon, Kolhapur, Hatk...","H.NO 586 R.S NO 87/2, RAM NAGAR VADGAON, KOLHA...",India,India,1
1,S1-323037216,S3-784663097,Digital Galaxy Ventures Pvt Ltd,Digital Pvt-Ventures Galaxy Ltd,"Plot No 10/2, Rajeev, Gruhakalpa, Balaji Nagar...",NaN,India,India,1
2,S1-584425961,S3-899116495,Advanced Chemical Global PC,Hyacinth's Textiles,"701 Harvard Lane, Newark, DE","9919 Brightling Ln, Austin, Texas",US,US,0
3,S1-49816297,S2-474362033,Vision Consultants Limited,Mr VCONSULTANTS.COM,"At Post Dream Land City Sawangi, Sawangi, Ward...","H.NO 0813 AT POST DREAM LAND CITY SAWANGI, BUL...",India,India,1
4,S1-205432256,S2-128286850,Chiropractic Group,Chnropracic Group,"7811 Ardmore Avenue, Parkville, MD","7811 ARDMORE AVE, PAKVILLE, MD",US,US,1
5,S1-703806829,S3-188026623,Premier Superior Metal Inc.,Premier Metal Inc. (Services),"OR, 3551 Reed Drive, Hillsboro, Fl 1","Oregon, Floor 1, Hillsboro, 3903 Reed Drive",US,US,1
6,S1-752372961,S2-166953481,Gleason Cardiology Inc,>> GLEASON CARDIOLOGY INCORPORATED,"56701 683 Road, Colcord, OK","0056701 683 RD, COLCORD, OK",US,US,1
7,S1-321086402,S3-208415333,C E & Z Micro,Shyam Technology Limited Partners,"5151 Guadalupe Road, Unit 3147, Phoenix, AZ","3Rd Floor, Thane, MH",US,India,0
8,S1-315209126,S3-332276263,OT Chemtech Pvt Ltd,Dermatology-5upreme Dermatology-5upreme Center,"Malapuram, Malappuram, Moyottil, Vairankode, T...","2441. Gunderson Ave, Berwyn, Illinois",India,US,0
9,S1-277319567,S2-331768869,TQ Kunj Private Limited,TQ Kunj Pnrne-Limited,"Muhusha, Po Khimlasa, Tehsil Khurai, District ...","PLTO 611 MUHUSHA, PO KHIMLASA, TEHSIL KHURAI, ...",India,India,1


In [23]:

def calculate_similarity(value1, value2):
    if pd.isna(value1) or pd.isna(value2):
        return 0.0

    value1 = str(value1).strip().lower()
    value2 = str(value2).strip().lower()

    if not value1 or not value2:
        return 0.0

    return ratio(value1, value2) / 100.0


# Create feature dataframe
features_df = training_pairs.copy()

# Name similarity
features_df["name_similarity"] = features_df.apply(
    lambda row: calculate_similarity(
        row["s1_name"],
        row["candidate_name"]
    ),
    axis=1
)

# Address similarity
features_df["address_similarity"] = features_df.apply(
    lambda row: calculate_similarity(
        row["s1_address"],
        row["candidate_address"]
    ),
    axis=1
)

# Missing-value indicators
features_df["candidate_name_missing"] = (
    features_df["candidate_name"].isna()
).astype(int)

features_df["candidate_address_missing"] = (
    features_df["candidate_address"].isna()
).astype(int)

# Country match
features_df["country_match"] = (
    features_df["s1_country"].fillna("").str.lower()
    ==
    features_df["candidate_country"].fillna("").str.lower()
).astype(int)

print("Feature engineering complete!")
print("Feature shape:", features_df.shape)

display(
    features_df[
        [
            "candidate_id",
            "name_similarity",
            "address_similarity",
            "candidate_name_missing",
            "candidate_address_missing",
            "country_match",
            "label"
        ]
    ].head(10)
)

Feature engineering complete!
Feature shape: (69536, 14)


,candidate_id,name_similarity,address_similarity,candidate_name_missing,candidate_address_missing,country_match,label
0,S2-777032501,0.093750,0.936170,0,0,1,1
1,S3-784663097,0.677419,0.000000,0,1,1,1
2,S3-899116495,0.260870,0.393443,0,0,1,0
3,S2-474362033,0.577778,0.848485,0,0,1,1
4,S2-128286850,0.914286,0.937500,0,0,1,1
5,S3-188026623,0.642857,0.506329,0,0,1,1
6,S2-166953481,0.785714,0.925926,0,0,1,1
7,S3-208415333,0.260870,0.317460,0,0,0,0
8,S3-332276263,0.246154,0.250000,0,0,0,0
9,S2-331768869,0.818182,0.952381,0,0,1,1


In [25]:
# Features used by the ML model
feature_columns = [
    "name_similarity",
    "address_similarity",
    "candidate_name_missing",
    "candidate_address_missing",
    "country_match"
]

X = features_df[feature_columns]
y = features_df["label"]

# Split into training and testing data
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples :", len(X_test))

print("\nTraining label distribution:")
print(y_train.value_counts())

print("\nTesting label distribution:")
print(y_test.value_counts())

Training samples: 55628
Testing samples : 13908

Training label distribution:
label
1    27814
0    27814
Name: count, dtype: int64

Testing label distribution:
label
1    6954
0    6954
Name: count, dtype: int64


In [27]:

# Create Random Forest model
model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    class_weight="balanced"
)

# Train the model
model.fit(X_train, y_train)

print("Model training completed!")

Model training completed!


In [34]:

dataframes = list(globals().items())

for name, value in dataframes:
    if isinstance(value, pd.DataFrame):
        print(f"{name} -> shape: {value.shape}")

train_s1 -> shape: (2206821, 4)
train_s2 -> shape: (5034616, 4)
train_s3 -> shape: (5285603, 4)
ground_truth -> shape: (2206821, 2)
sample_s1 -> shape: (10000, 4)
sample_ground_truth -> shape: (10000, 2)
s1_lookup -> shape: (2206821, 3)
s2_lookup -> shape: (5034616, 3)
s3_lookup -> shape: (5285603, 3)
positive_pairs_df -> shape: (34768, 9)
gt_match -> shape: (1, 2)
candidates -> shape: (3170056, 4)
selected -> shape: (2, 4)
s2_pool -> shape: (100000, 4)
s3_pool -> shape: (100000, 4)
candidate_pool -> shape: (200000, 4)
negative_candidate_pool -> shape: (199318, 4)
negative_candidates -> shape: (34768, 4)
s1_negative -> shape: (34768, 4)
negative_pairs_df -> shape: (34768, 9)
training_pairs -> shape: (69536, 9)
features_df -> shape: (69536, 14)
X -> shape: (69536, 5)
X_train -> shape: (55628, 5)
X_test -> shape: (13908, 5)


In [36]:
# Find the feature dataframe automatically
feature_df = None

for name, value in list(globals().items()):
    if isinstance(value, pd.DataFrame):
        if value.shape == (69536, 14):
            feature_df = value
            print("Feature dataframe found:", name)
            break

if feature_df is None:
    print("Feature dataframe with shape (69536, 14) not found.")
else:
    feature_columns = [
        "name_similarity",
        "address_similarity",
        "candidate_name_missing",
        "candidate_address_missing",
        "country_match"
    ]

    X = feature_df[feature_columns]
    y = feature_df["label"]

    print("\nFeature matrix:", X.shape)
    print("Target:", y.shape)

    from sklearn.model_selection import train_test_split

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.20,
        random_state=42,
        stratify=y
    )

    print("\n================================")
    print("TRAIN / TEST DATA READY")
    print("================================")
    print("Training samples:", len(X_train))
    print("Testing samples :", len(X_test))

    print("\nTraining labels:")
    print(y_train.value_counts())

    print("\nTesting labels:")
    print(y_test.value_counts())

Feature dataframe found: features_df

Feature matrix: (69536, 5)
Target: (69536,)

TRAIN / TEST DATA READY
Training samples: 55628
Testing samples : 13908

Training labels:
label
1    27814
0    27814
Name: count, dtype: int64

Testing labels:
label
1    6954
0    6954
Name: count, dtype: int64


In [ ]:

# TRAIN MACHINE LEARNING MODEL

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print("Training Random Forest model...")

model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced"
)

model.fit(X_train, y_train)

print("================================")
print("MODEL TRAINING COMPLETE")
print("================================")

Training Random Forest model...
MODEL TRAINING COMPLETE


In [ ]:

# MODEL EVALUATION

# Make predictions
y_pred = model.predict(X_test)

# Accuracy
accuracy = accuracy_score(y_test, y_pred)

print("================================")
print("MODEL EVALUATION")
print("================================")

print("Accuracy :", accuracy)

# Classification report
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

# Confusion Matrix
print("\nConfusion Matrix:")
cm = confusion_matrix(y_test, y_pred)
print(cm)

MODEL EVALUATION
Accuracy : 0.9867702041990222

Classification Report:
              precision    recall  f1-score   support

           0       0.98      0.99      0.99      6954
           1       0.99      0.98      0.99      6954

    accuracy                           0.99     13908
   macro avg       0.99      0.99      0.99     13908
weighted avg       0.99      0.99      0.99     13908


Confusion Matrix:
[[6884   70]
 [ 114 6840]]


In [39]:
# ============================================
# FEATURE IMPORTANCE
# ============================================

importance_df = pd.DataFrame({
    "Feature": feature_columns,
    "Importance": model.feature_importances_
})

importance_df = importance_df.sort_values(
    by="Importance",
    ascending=False
)

print("================================")
print("FEATURE IMPORTANCE")
print("================================")

display(importance_df)

FEATURE IMPORTANCE


,Feature,Importance
1,address_similarity,0.522659
0,name_similarity,0.385067
4,country_match,0.085079
3,candidate_address_missing,0.007195
2,candidate_name_missing,0.000000


In [42]:

print("Available DataFrames:")

for name, value in list(globals().items()):
    if isinstance(value, pd.DataFrame):
        print(name, "->", value.shape)

Available DataFrames:
train_s1 -> (2206821, 4)
train_s2 -> (5034616, 4)
train_s3 -> (5285603, 4)
ground_truth -> (2206821, 2)
sample_s1 -> (10000, 4)
sample_ground_truth -> (10000, 2)
s1_lookup -> (2206821, 3)
s2_lookup -> (5034616, 3)
s3_lookup -> (5285603, 3)
positive_pairs_df -> (34768, 9)
gt_match -> (1, 2)
candidates -> (3170056, 4)
selected -> (2, 4)
s2_pool -> (100000, 4)
s3_pool -> (100000, 4)
candidate_pool -> (200000, 4)
negative_candidate_pool -> (199318, 4)
negative_candidates -> (34768, 4)
s1_negative -> (34768, 4)
negative_pairs_df -> (34768, 9)
training_pairs -> (69536, 9)
features_df -> (69536, 14)
X -> (69536, 5)
X_train -> (55628, 5)
X_test -> (13908, 5)
value -> (69536, 14)
feature_df -> (69536, 14)
importance_df -> (5, 2)


In [44]:
# Check S1 dataframe
print(train_s1.shape)
display(train_s1.head())

(2206821, 4)


,entity_id,business_name,business_address,country
0,S1-925783039,Orelee's Barbershop,"1795 Westchester Drive, High Point, NC",US
1,S1-773889195,Prime Money,"17560 Ellis Road, Tahlequah, OK",US
2,S1-377745466,B+ Retail Inc,"1712 Montebello Avenue, Phoenix, AZ",US
3,S1-133037285,Christ Chapel,"2100 Cameron Drive, Unit APARTMENT G, Dundalk, MD",US
4,S1-755362802,Prabhav Business Center,"797, Lake Town Block A, Kolkata, Howrah, West ...",India


In [ ]:

# FINAL ENTITY MATCHING DEMO


demo_s1_id = "S1-965667"

# Get S1 record
s1_demo = train_s1[
    train_s1["entity_id"] == demo_s1_id
].iloc[0]

print("==============================================")
print("SOURCE 1 RECORD")
print("==============================================")
print("Entity ID :", s1_demo["entity_id"])
print("Name      :", s1_demo["business_name"])
print("Address   :", s1_demo["business_address"])
print("Country   :", s1_demo["country"])



# GET CANDIDATES


s2_demo = train_s2[
    train_s2["country"] == s1_demo["country"]
].sample(
    n=20,
    random_state=42
)

s3_demo = train_s3[
    train_s3["country"] == s1_demo["country"]
].sample(
    n=20,
    random_state=42
)

candidates_demo = pd.concat(
    [s2_demo, s3_demo],
    ignore_index=True
)


# CALCULATE FEATURES
demo_results = []

for _, candidate in candidates_demo.iterrows():

    name_score = calculate_similarity(
        s1_demo["business_name"],
        candidate["business_name"]
    )

    address_score = calculate_similarity(
        s1_demo["business_address"],
        candidate["business_address"]
    )

    name_missing = int(
        pd.isna(candidate["business_name"])
    )

    address_missing = int(
        pd.isna(candidate["business_address"])
    )

    country_match = int(
        str(s1_demo["country"]).lower()
        ==
        str(candidate["country"]).lower()
    )

    demo_results.append({
        "candidate_id": candidate["entity_id"],
        "candidate_name": candidate["business_name"],
        "candidate_address": candidate["business_address"],
        "name_similarity": name_score,
        "address_similarity": address_score,
        "candidate_name_missing": name_missing,
        "candidate_address_missing": address_missing,
        "country_match": country_match
    })


# Convert to DataFrame
demo_features = pd.DataFrame(demo_results)


# MODEL PREDICTION

X_demo = demo_features[feature_columns]

demo_features["prediction"] = model.predict(X_demo)

demo_features["match_probability"] = (
    model.predict_proba(X_demo)[:, 1]
)


# SORT BY MATCH PROBABILITY

demo_results_final = demo_features.sort_values(
    by="match_probability",
    ascending=False
)


print("\n==============================================")
print("TOP MATCHING CANDIDATES")
print("==============================================")

display(
    demo_results_final[
        [
            "candidate_id",
            "candidate_name",
            "name_similarity",
            "address_similarity",
            "country_match",
            "match_probability",
            "prediction"
        ]
    ].head(10)
)

SOURCE 1 RECORD
Entity ID : S1-965667
Name      : Maure Williams Colombier Inc
Address   : 85 Wayne Avenue, Ticonderoga, NY
Country   : US

TOP MATCHING CANDIDATES


,candidate_id,candidate_name,name_similarity,address_similarity,country_match,match_probability,prediction
3,S2-687069736,Center Horizon Neuberger XR Westgate,0.250000,0.535211,1,0.19,0
36,S3-240385228,Clean Plus Inc,0.380952,0.468750,1,0.03,0
26,S3-105889526,Wells Immersion Lakeside,0.377358,0.467532,1,0.01,0
0,S2-744400191,Optimal Mndgal Dynamics,0.313725,0.463768,1,0.00,0
4,S2-98171065,BRIGHT PIZZA! INC,0.355556,0.266667,1,0.00,0
5,S2-893246324,Supreme Services,0.409091,0.338983,1,0.00,0
1,S2-591196144,LOUISVILLE URGENT CARE ALL HEALTH | www.louisv...,0.317073,0.324324,1,0.00,0
2,S2-276454667,Glass and Buckman,0.311111,0.457143,1,0.00,0
7,S2-61220223,oudefense.com,0.292683,0.333333,1,0.00,0
6,S2-91590235,Helge Sosa Xrp Ltd.,0.255319,0.376812,1,0.00,0


In [ ]:

# VERIFY ACTUAL MATCHES FOR DEMO RECORD

demo_s1_id = "S1-965667"

print("==============================================")
print("GROUND TRUTH MATCH VERIFICATION")
print("==============================================")

# Get ground truth row
gt_row = ground_truth[
    ground_truth["source1_entity_id"] == demo_s1_id
]

if len(gt_row) == 0:
    print("No ground truth found for:", demo_s1_id)

else:
    matched_ids = str(
        gt_row.iloc[0]["matched_entity_ids"]
    ).split(",")

    print("S1 Entity ID :", demo_s1_id)
    print("Number of actual matches :", len(matched_ids))

    print("\nActual matched candidate IDs:")
    for candidate_id in matched_ids:
        print(candidate_id)

GROUND TRUTH MATCH VERIFICATION
S1 Entity ID : S1-965667
Number of actual matches : 5

Actual matched candidate IDs:
S2-681193310
S2-743505751
S3-775321672
S3-11291185
S3-860443364


In [ ]:

# FINAL ACTUAL MATCH TEST

demo_s1_id = "S1-965667"

# Get S1 record
s1_demo = train_s1[
    train_s1["entity_id"] == demo_s1_id
].iloc[0]

# Get actual Ground Truth matches
gt_row = ground_truth[
    ground_truth["source1_entity_id"] == demo_s1_id
].iloc[0]

actual_match_ids = str(
    gt_row["matched_entity_ids"]
).split(",")

print("==============================================")
print("FINAL ENTITY MATCH TEST")
print("==============================================")

print("Source 1 ID :", demo_s1_id)
print("Name        :", s1_demo["business_name"])
print("Address     :", s1_demo["business_address"])
print("Country     :", s1_demo["country"])

print("\nActual Ground Truth Matches:")
for match_id in actual_match_ids:
    print("  ", match_id)
# Get actual matching records from S2 and S3

actual_s2 = train_s2[
    train_s2["entity_id"].isin(actual_match_ids)
].copy()

actual_s3 = train_s3[
    train_s3["entity_id"].isin(actual_match_ids)
].copy()

actual_candidates = pd.concat(
    [actual_s2, actual_s3],
    ignore_index=True
)


# --------------------------------------------
# Calculate features
# --------------------------------------------

final_results = []

for _, candidate in actual_candidates.iterrows():

    name_score = calculate_similarity(
        s1_demo["business_name"],
        candidate["business_name"]
    )

    address_score = calculate_similarity(
        s1_demo["business_address"],
        candidate["business_address"]
    )

    name_missing = int(
        pd.isna(candidate["business_name"])
    )

    address_missing = int(
        pd.isna(candidate["business_address"])
    )

    country_match = int(
        str(s1_demo["country"]).lower()
        ==
        str(candidate["country"]).lower()
    )

    final_results.append({
        "candidate_id": candidate["entity_id"],
        "candidate_name": candidate["business_name"],
        "candidate_address": candidate["business_address"],
        "name_similarity": name_score,
        "address_similarity": address_score,
        "candidate_name_missing": name_missing,
        "candidate_address_missing": address_missing,
        "country_match": country_match
    })


final_df = pd.DataFrame(final_results)

# Model prediction
X_final = final_df[feature_columns]

final_df["match_probability"] = (
    model.predict_proba(X_final)[:, 1]
)

final_df["prediction"] = model.predict(X_final)

final_df["ground_truth"] = 1


# Sort by probability
final_df = final_df.sort_values(
    by="match_probability",
    ascending=False
)


print("\n==============================================")
print("MODEL SCORES FOR ACTUAL MATCHES")
print("==============================================")

display(
    final_df[
        [
            "candidate_id",
            "candidate_name",
            "name_similarity",
            "address_similarity",
            "country_match",
            "match_probability",
            "prediction",
            "ground_truth"
        ]
    ]
)

FINAL ENTITY MATCH TEST
Source 1 ID : S1-965667
Name        : Maure Williams Colombier Inc
Address     : 85 Wayne Avenue, Ticonderoga, NY
Country     : US

Actual Ground Truth Matches:
   S2-681193310
   S2-743505751
   S3-775321672
   S3-11291185
   S3-860443364

MODEL SCORES FOR ACTUAL MATCHES


,candidate_id,candidate_name,name_similarity,address_similarity,country_match,match_probability,prediction,ground_truth
0,S2-681193310,Maure Wilblims Colombier Inc,0.964286,0.000000,1,1.000,1,1
1,S2-743505751,Maure Williams Colombier,0.923077,0.000000,1,1.000,1,1
3,S3-775321672,Dréxkor,0.171429,0.775000,1,1.000,1,1
4,S3-11291185,maurewilliamscolombier.com,0.851852,0.702703,1,1.000,1,1
2,S3-860443364,Maure Williams Inc Center,0.679245,0.000000,1,0.998,1,1


In [ ]:

# REUSABLE ENTITY MATCHING FUNCTION

def find_entity_matches(s1_entity_id, top_n=10):

    # Get Source 1 record

    s1_rows = train_s1[
        train_s1["entity_id"] == s1_entity_id
    ]

    if len(s1_rows) == 0:
        print("S1 entity not found:", s1_entity_id)
        return None

    s1_record = s1_rows.iloc[0]

    print("==============================================")
    print("ENTITY MATCHING RESULT")
    print("==============================================")

    print("Source 1 ID :", s1_record["entity_id"])
    print("Name        :", s1_record["business_name"])
    print("Address     :", s1_record["business_address"])
    print("Country     :", s1_record["country"])


    # Candidate selection

    # First filter by country
    s2_candidates = train_s2[
        train_s2["country"] == s1_record["country"]
    ]

    s3_candidates = train_s3[
        train_s3["country"] == s1_record["country"]
    ]


    # Limit candidates for demo speed
    s2_candidates = s2_candidates.sample(
        n=min(1000, len(s2_candidates)),
        random_state=42
    )

    s3_candidates = s3_candidates.sample(
        n=min(1000, len(s3_candidates)),
        random_state=42
    )


    candidates = pd.concat(
        [s2_candidates, s3_candidates],
        ignore_index=True
    )


    
    # Feature calculation

    results = []

    for _, candidate in candidates.iterrows():

        name_score = calculate_similarity(
            s1_record["business_name"],
            candidate["business_name"]
        )

        address_score = calculate_similarity(
            s1_record["business_address"],
            candidate["business_address"]
        )

        name_missing = int(
            pd.isna(candidate["business_name"])
        )

        address_missing = int(
            pd.isna(candidate["business_address"])
        )

        country_match = int(
            str(s1_record["country"]).lower()
            ==
            str(candidate["country"]).lower()
        )

        results.append({
            "candidate_id": candidate["entity_id"],
            "candidate_name": candidate["business_name"],
            "candidate_address": candidate["business_address"],
            "name_similarity": name_score,
            "address_similarity": address_score,
            "candidate_name_missing": name_missing,
            "candidate_address_missing": address_missing,
            "country_match": country_match
        })


    results_df = pd.DataFrame(results)

    # Model prediction
    X_candidates = results_df[feature_columns]

    results_df["match_probability"] = (
        model.predict_proba(X_candidates)[:, 1]
    )

    results_df["prediction"] = model.predict(
        X_candidates
    )

    # Keep only predicted matches
    matches = results_df[
        results_df["prediction"] == 1
    ].sort_values(
        by="match_probability",
        ascending=False
    )


    print("\n==============================================")
    print("TOP PREDICTED MATCHES")
    print("==============================================")

    display(
        matches[
            [
                "candidate_id",
                "candidate_name",
                "name_similarity",
                "address_similarity",
                "country_match",
                "match_probability",
                "prediction"
            ]
        ].head(top_n)
    )

    return matches

In [49]:
# Test the reusable matching function

result = find_entity_matches(
    "S1-965667",
    top_n=10
)

ENTITY MATCHING RESULT
Source 1 ID : S1-965667
Name        : Maure Williams Colombier Inc
Address     : 85 Wayne Avenue, Ticonderoga, NY
Country     : US

TOP PREDICTED MATCHES


,candidate_id,candidate_name,name_similarity,address_similarity,country_match,match_probability,prediction
930,S2-989687758,Fluxdova,0.111111,0.612903,1,1.00,1
1956,S3-70512582,Kelley Wroldaie Inc,0.510638,0.492308,1,0.86,1
318,S2-619155410,CZB [Shreya],0.150000,0.535211,1,0.85,1
308,S2-996255772,Sterling LLC,0.300000,0.584615,1,0.75,1
747,S2-122886753,Continental Ribbon Highland Integrated,0.303030,0.500000,1,0.63,1
1581,S3-373062520,@2020eye,0.111111,0.444444,1,0.61,1
43,S2-471085924,Group Fredra's Dynamic Fashion,0.379310,0.476190,1,0.59,1
508,S2-36813606,Ziex Abrdn Corp,0.325581,0.579710,1,0.58,1
537,S2-806597619,"Group Ayres, Benitez and Badgley",0.333333,0.571429,1,0.54,1


In [ ]:

# IMPROVED ENTITY MATCHING

def find_entity_matches(s1_entity_id, top_n=10):

    s1_rows = train_s1[
        train_s1["entity_id"] == s1_entity_id
    ]

    if len(s1_rows) == 0:
        print("S1 entity not found:", s1_entity_id)
        return None

    s1_record = s1_rows.iloc[0]

    print("==============================================")
    print("IMPROVED ENTITY MATCHING")
    print("==============================================")

    print("Source 1 ID :", s1_entity_id)
    print("Name        :", s1_record["business_name"])
    print("Address     :", s1_record["business_address"])
    print("Country     :", s1_record["country"])

    # Country filtering
    s2_candidates = train_s2[
        train_s2["country"] == s1_record["country"]
    ].copy()

    s3_candidates = train_s3[
        train_s3["country"] == s1_record["country"]
    ].copy()

    candidates = pd.concat(
        [s2_candidates, s3_candidates],
        ignore_index=True
    )

    # Calculate similarity for candidate pool

    candidate_results = []

    for _, candidate in candidates.iterrows():

        name_score = calculate_similarity(
            s1_record["business_name"],
            candidate["business_name"]
        )

        address_score = calculate_similarity(
            s1_record["business_address"],
            candidate["business_address"]
        )

        candidate_results.append({
            "candidate_id": candidate["entity_id"],
            "candidate_name": candidate["business_name"],
            "candidate_address": candidate["business_address"],
            "name_similarity": name_score,
            "address_similarity": address_score,
            "candidate_name_missing": int(
                pd.isna(candidate["business_name"])
            ),
            "candidate_address_missing": int(
                pd.isna(candidate["business_address"])
            ),
            "country_match": int(
                str(s1_record["country"]).lower()
                ==
                str(candidate["country"]).lower()
            )
        })


    candidates_df = pd.DataFrame(candidate_results)


   
    # Candidate filtering
    # Keep strongest similarity candidates

    candidates_df["combined_similarity"] = (
        0.5 * candidates_df["name_similarity"]
        +
        0.5 * candidates_df["address_similarity"]
    )

    candidates_df = candidates_df.sort_values(
        by="combined_similarity",
        ascending=False
    ).head(200)
    # Random Forest prediction
   

    X_candidates = candidates_df[feature_columns]

    candidates_df["match_probability"] = (
        model.predict_proba(X_candidates)[:, 1]
    )

    candidates_df["prediction"] = model.predict(
        X_candidates
    )


    
    # Final ranking
    

    final_matches = candidates_df.sort_values(
        by="match_probability",
        ascending=False
    ).head(top_n)


    print("\n==============================================")
    print("TOP MATCHING CANDIDATES")
    print("==============================================")

    display(
        final_matches[
            [
                "candidate_id",
                "candidate_name",
                "name_similarity",
                "address_similarity",
                "country_match",
                "match_probability",
                "prediction"
            ]
        ]
    )

    return final_matches

In [ ]:

# FINAL MODEL SUMMARY


from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

# Predictions on test data
y_pred = model.predict(X_test)

# Calculate metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print("==============================================")
print("FINAL ENTITY MATCHING MODEL SUMMARY")
print("==============================================")

print(f"Training samples : {len(X_train)}")
print(f"Testing samples  : {len(X_test)}")

print("\nModel : Random Forest")

print("\n==============================================")
print("PERFORMANCE")
print("==============================================")

print(f"Accuracy  : {accuracy:.4f}")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1 Score  : {f1:.4f}")

print("\n==============================================")
print("FEATURES USED")
print("==============================================")

for feature in feature_columns:
    print("-", feature)

print("\n==============================================")
print("PROJECT STATUS")
print("==============================================")

print("Data loading              : COMPLETE")
print("Positive pair generation  : COMPLETE")
print("Negative pair generation  : COMPLETE")
print("Feature engineering       : COMPLETE")
print("Model training            : COMPLETE")
print("Model evaluation          : COMPLETE")
print("Feature importance        : COMPLETE")
print("Entity matching demo      : COMPLETE")
print("Ground truth validation   : COMPLETE")

print("\n==============================================")
print("ENTITY MATCHING PIPELINE COMPLETE")
print("==============================================")

FINAL ENTITY MATCHING MODEL SUMMARY
Training samples : 55628
Testing samples  : 13908

Model : Random Forest

PERFORMANCE
Accuracy  : 0.9868
Precision : 0.9899
Recall    : 0.9836
F1 Score  : 0.9867

FEATURES USED
- name_similarity
- address_similarity
- candidate_name_missing
- candidate_address_missing
- country_match

PROJECT STATUS
Data loading              : COMPLETE
Positive pair generation  : COMPLETE
Negative pair generation  : COMPLETE
Feature engineering       : COMPLETE
Model training            : COMPLETE
Model evaluation          : COMPLETE
Feature importance        : COMPLETE
Entity matching demo      : COMPLETE
Ground truth validation   : COMPLETE

ENTITY MATCHING PIPELINE COMPLETE
